# Phase 1D -- Graph Engine, Full Build: RGCN, CARE-GNN, Leaderboard, Expanded Search

Implements `docs/PHASE_1D_GRAPH_ENGINE_FULL.md`. Generated from `notebooks/src/02_graph_full.py` by
`scripts/build_notebooks.py` -- edit the source, not this .ipynb.

1. **Setup + reproduction check** -- same data loading, temporal split, class weighting, and
   training protocol as `01_graph_baseline.ipynb`. XGBoost (Tier-1, Tier-1+2) and the Phase 1C
   wide-search GraphSAGE are refit from their stored best hyperparameters, and checked against the
   earlier runs' numbers, so the Phase 1D leaderboard compares like with like.
2. **RGCN** with label-free relation types (payment direction; direction x hub), plus an untyped
   single-relation control on the same edges.
3. **CARE-GNN** (similarity-aware neighbor selection + the paper's threshold update), with a
   before/after comparison on the Phase 1C camouflaged subset at a matched false-positive rate.
4. **Leaderboard** of every model trained in this notebook (the full Phase 1-1D leaderboard is
   merged locally from both results files by `scripts/build_graph_leaderboard.py`).
5. **Expanded Optuna search** (up to 100 trials, time-boxed) on the best GNN.
6. **3-seed robustness check** on the final best GNN.
7. **Closing analysis** and save.

Every task saves to `artifacts/graph/results_phase1d.json` as soon as it finishes.

**Kaggle:** Accelerator = GPU, Internet = on, Run All. Expected runtime: roughly 1-5 hours, dominated
by the expanded search, which is capped at 4 hours (`SEARCH_TIMEOUT_SEC`).

## Setup

In [ ]:
# Kaggle: Accelerator = GPU, Internet = on.
import os
import subprocess
import sys

# True only when tests/smoke_notebook_02.py runs this notebook's code on synthetic data on a CPU.
SMOKE_TEST = os.environ.get("TRUSTMESH_SMOKE") == "1"


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


if not SMOKE_TEST:
    for module_name, package in [("torch_geometric", "torch_geometric"), ("xgboost", "xgboost"),
                                 ("optuna", "optuna"), ("networkx", "networkx")]:
        try:
            __import__(module_name)
        except ImportError:
            pip_install(package)

In [ ]:
import json
import math
import time
from pathlib import Path

import numpy as np
import optuna
import torch

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_ROOT = Path("./data/elliptic")
OUTPUT_DIR = Path(os.environ.get("TRUSTMESH_OUTPUT_DIR", "./artifacts/graph"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() and not SMOKE_TEST else "cpu")

DEFAULT_EPOCHS = 5 if SMOKE_TEST else 40          # "defaults" protocol, as for GraphSAGE/GAT/GIN in notebook 01
REPRO_REFIT_EPOCHS = 20 if SMOKE_TEST else 150    # notebook 01's early-stopped refit budget
REFIT_MAX_EPOCHS = 20 if SMOKE_TEST else 300      # early-stopped refits in this notebook (patience 15)
SEARCH_TRIALS = 3 if SMOKE_TEST else 100
SEARCH_MAX_EPOCHS = 20 if SMOKE_TEST else 300
SEARCH_TIMEOUT_SEC = 120 if SMOKE_TEST else 4 * 3600
ROBUSTNESS_SEEDS = [SEED, SEED + 1, SEED + 2]
CAMOUFLAGE_TARGET_FPR = 0.05

# Real numbers from earlier Kaggle runs (artifacts/graph/results.json). Used ONLY to check that this
# notebook reproduces notebook 01's setup; they are not Phase 1D results.
REFERENCE = {
    "split_counts": [26905, 4330, 15329],
    "pos_weight": 8.107989167230873,
    "xgboost_tier1": 0.7447529434085257,
    "xgboost_tier12": 0.7474345053830228,
    "graphsage_wide_search": 0.5770002559534624,
    "xgboost_tier12_3seed_mean": 0.7482402411555441,
    "xgboost_tier12_3seed_std": 0.003011179506437757,
    "graphsage_default_directed": 0.5114779015333686,
}
# Best config from notebook 01's 50-trial wider GraphSAGE search (Phase 1C).
GRAPHSAGE_WIDE_PARAMS = {"hidden_dim": 256, "lr": 0.001031859734651928, "dropout": 0.22808425766001517}

RESULTS_PATH = OUTPUT_DIR / "results_phase1d.json"
RESULTS = {"smoke_test": SMOKE_TEST}


def save_progress(key, value):
    RESULTS[key] = value
    RESULTS_PATH.write_text(json.dumps(RESULTS, indent=2), encoding="utf-8")
    print(f"[progress] saved '{key}' -> {RESULTS_PATH}")


optuna.logging.set_verbosity(optuna.logging.WARNING)
print("device:", DEVICE, "| SMOKE_TEST:", SMOKE_TEST, "| torch", torch.__version__)

## Shared graph-engine code (inlined from `engines/graph/`)

In [ ]:
# ---- inlined from engines/graph/data.py (edit the module, then rebuild) ----
"""Elliptic loading and the temporal split shared by every graph-engine model."""
import os

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.datasets import EllipticBitcoinDataset

EXPECTED_NODES = 203769
EXPECTED_FEATURES = 165
N_LOCAL_FEATURES = 93  # features 0-92 are local; 93-164 are precomputed 1-hop neighborhood aggregates
LICIT, ILLICIT, UNKNOWN = 0, 1, 2
TRAIN_STEPS = (1, 30)
VAL_STEPS = (31, 35)
TEST_STEPS = (36, 49)


def load_elliptic(root):
    dataset = EllipticBitcoinDataset(root=str(root))
    data = dataset[0]
    assert data.num_nodes == EXPECTED_NODES, f"unexpected node count: {data.num_nodes}"
    assert data.num_node_features == EXPECTED_FEATURES, f"unexpected feature count: {data.num_node_features}"

    # PyG drops txId and time_step when it builds data.x, so the time step can't be read from x
    # (data.x[:, 0] is a normalized feature). Take it from the raw features CSV, whose row order
    # matches data.x / data.y.
    features_csv = next(p for p in dataset.raw_paths if "feat" in os.path.basename(p).lower())
    steps = pd.read_csv(features_csv, header=None, usecols=[1]).iloc[:, 0].to_numpy()
    data.time_step = torch.from_numpy(steps).long()
    assert len(data.time_step) == data.num_nodes, "time_step row count doesn't match data.x"
    return data


def make_synthetic_elliptic(nodes_per_step=20, num_features=EXPECTED_FEATURES, seed=0):
    """Tiny Elliptic-shaped graph for local smoke tests only -- never used for reported numbers.
    Same attributes and label coding as `load_elliptic`, edges only within a time step (like
    Elliptic), a few hub nodes, and label/homophily signal so models have something to learn."""
    g = torch.Generator().manual_seed(seed)
    num_steps = TEST_STEPS[1]
    n = nodes_per_step * num_steps
    time_step = torch.arange(1, num_steps + 1).repeat_interleave(nodes_per_step)

    y = torch.full((n,), UNKNOWN, dtype=torch.long)
    r = torch.rand(n, generator=g)
    y[r < 0.40] = LICIT
    y[r < 0.12] = ILLICIT

    x = torch.randn(n, num_features, generator=g)
    x[:, :5] += 1.5 * (y == ILLICIT).float().unsqueeze(1)

    src, dst = [], []
    for t in range(1, num_steps + 1):
        idx = (time_step == t).nonzero(as_tuple=True)[0]
        hub = idx[0]
        for i in idx.tolist():
            if i != hub.item():
                src.append(i)
                dst.append(hub.item())
        for _ in range(len(idx)):
            a, b = idx[torch.randint(len(idx), (2,), generator=g)].tolist()
            if a != b and (y[a] == y[b] or torch.rand(1, generator=g).item() < 0.3):
                src.append(a)
                dst.append(b)
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y, time_step=time_step)


def temporal_masks(data):
    """Train / val / test masks over labeled nodes only (unknowns have no ground truth)."""
    labeled = data.y != UNKNOWN
    t = data.time_step

    def steps(lo, hi):
        return labeled & (t >= lo) & (t <= hi)

    return steps(*TRAIN_STEPS), steps(*VAL_STEPS), steps(*TEST_STEPS)


def class_pos_weight(y, train_mask):
    y_train = y[train_mask].cpu().numpy()
    return float((y_train == LICIT).sum() / max((y_train == ILLICIT).sum(), 1))


def split_counts(train_mask, val_mask, test_mask):
    return tuple(int(m.sum()) for m in (train_mask, val_mask, test_mask))


def edge_time_stats(edge_index, time_step):
    """How many edges connect nodes in the same time step vs. different steps."""
    ei = edge_index.cpu()
    same = int((time_step[ei[0]] == time_step[ei[1]]).sum())
    total = int(ei.size(1))
    return {"edges": total, "same_step": same, "cross_step": total - same}


def to_numpy_features(data):
    return data.x.cpu().numpy().astype(np.float32)

In [ ]:
# ---- inlined from engines/graph/features.py (edit the module, then rebuild) ----
"""Tier-2 classical graph features (Phase 1B ablation), computed exactly as in notebook 01."""
import networkx as nx
import numpy as np
from torch_geometric.utils import degree, to_undirected

TIER2_FEATURE_NAMES = ["degree", "out_in_degree_ratio", "pagerank", "clustering"]


def tier2_graph_features(edge_index, num_nodes):
    ei = edge_index.cpu()
    ei_u = to_undirected(ei)
    degree_arr = degree(ei_u[0], num_nodes=num_nodes).numpy()

    g_undirected = nx.Graph()
    g_undirected.add_nodes_from(range(num_nodes))
    g_undirected.add_edges_from(ei_u.t().numpy().tolist())

    g_directed = nx.DiGraph()
    g_directed.add_nodes_from(range(num_nodes))
    g_directed.add_edges_from(ei.t().numpy().tolist())

    in_deg = dict(g_directed.in_degree())
    out_deg = dict(g_directed.out_degree())
    in_arr = np.array([in_deg[i] for i in range(num_nodes)], dtype=np.float32)
    out_arr = np.array([out_deg[i] for i in range(num_nodes)], dtype=np.float32)
    io_ratio = (out_arr + 1.0) / (in_arr + 1.0)  # +1 smoothing avoids division by zero

    pagerank = nx.pagerank(g_directed, tol=1e-4, max_iter=100)
    pagerank_arr = np.array([pagerank[i] for i in range(num_nodes)], dtype=np.float32)

    clustering = nx.clustering(g_undirected)
    clustering_arr = np.array([clustering[i] for i in range(num_nodes)], dtype=np.float32)

    return np.stack([degree_arr, io_ratio, pagerank_arr, clustering_arr], axis=1)

In [ ]:
# ---- inlined from engines/graph/baseline.py (edit the module, then rebuild) ----
"""XGBoost baselines, refit from the best hyperparameters found by the Phase 1 / 1B Optuna searches."""
import xgboost as xgb

# Best params from the Optuna searches in notebooks/01_graph_baseline.ipynb (recorded in
# artifacts/graph/results.json). scale_pos_weight is recomputed from the training split, not stored.
XGB_TIER1_PARAMS = {
    "max_depth": 7,
    "eta": 0.11098926863362221,
    "subsample": 0.8454933633696089,
    "colsample_bytree": 0.7300523345284323,
    "min_child_weight": 5,
}
XGB_TIER12_PARAMS = {
    "max_depth": 4,
    "eta": 0.0858855758022927,
    "subsample": 0.7582529708241003,
    "colsample_bytree": 0.9860930071516779,
    "min_child_weight": 3,
}


def fit_xgb(X_train, y_train, params, pos_weight, seed):
    model = xgb.XGBClassifier(
        **params,
        scale_pos_weight=pos_weight,
        objective="binary:logistic",
        eval_metric="aucpr",
        seed=seed,
        n_estimators=300,
    )
    model.fit(X_train, y_train)
    return model

In [ ]:
# ---- inlined from engines/graph/relations.py (edit the module, then rebuild) ----
"""Relation typing for Elliptic's untyped edges (RGCN and CARE-GNN).

Elliptic edges are directed payment flows: u -> v means an output of transaction u is spent by v.
The GraphSAGE / GAT / GIN models in notebook 01 passed messages only along that direction, so
each node saw its upstream (payer-side) neighbors and never its downstream ones.

Schemes (all label-free and available at inference time, so no label leakage into the graph):
  single     -- both directions, one relation. Control: same edges as `direction`, no typing.
  direction  -- relation 0: message from an upstream neighbor (original edge u -> v)
                relation 1: message from a downstream neighbor (reversed edge v -> u)
  direction_hub -- `direction` x whether the message source is a hub (undirected degree at or
                above the given percentile), 4 relations. Hubs are likely services such as
                exchanges or mixers, whose neighbors may be less informative.
"""
import torch
from torch_geometric.utils import degree, to_undirected

RELATION_SCHEMES = ("single", "direction", "direction_hub")


def hub_degree_threshold(edge_index, num_nodes, percentile=99.0):
    deg = degree(to_undirected(edge_index.cpu())[0], num_nodes=num_nodes)
    return float(torch.quantile(deg, percentile / 100.0))


def build_relations(edge_index, num_nodes, scheme, hub_percentile=99.0):
    """Returns (edge_index, edge_type, relation_names, info)."""
    ei = edge_index.cpu()
    rev = ei.flip(0)
    both = torch.cat([ei, rev], dim=1)
    direction = torch.cat([torch.zeros(ei.size(1), dtype=torch.long), torch.ones(rev.size(1), dtype=torch.long)])

    if scheme == "single":
        return both, torch.zeros(both.size(1), dtype=torch.long), ["any"], {}
    if scheme == "direction":
        return both, direction, ["from_upstream", "from_downstream"], {}
    if scheme == "direction_hub":
        threshold = hub_degree_threshold(ei, num_nodes, hub_percentile)
        deg = degree(to_undirected(ei)[0], num_nodes=num_nodes)
        source_is_hub = (deg[both[0]] >= threshold).long()
        edge_type = direction * 2 + source_is_hub
        names = ["from_upstream", "from_upstream_hub", "from_downstream", "from_downstream_hub"]
        info = {
            "hub_percentile": hub_percentile,
            "hub_degree_threshold": threshold,
            "hub_nodes": int((deg >= threshold).sum()),
        }
        return both, edge_type, names, info
    raise ValueError(f"unknown relation scheme: {scheme}")


def relation_counts(edge_type, relation_names):
    counts = torch.bincount(edge_type, minlength=len(relation_names))
    return {name: int(c) for name, c in zip(relation_names, counts)}

In [ ]:
# ---- inlined from engines/graph/gnn.py (edit the module, then rebuild) ----
"""Message-passing GNNs: GraphSAGE (same architecture as notebook 01) and RGCN."""
import torch
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv, SAGEConv


class GraphSAGEFlex(torch.nn.Module):
    """2-layer GraphSAGE, identical to GraphSAGEFlex in notebooks/01_graph_baseline.ipynb."""

    def __init__(self, in_channels, hidden_channels=64, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index).squeeze(-1)


class RGCN(torch.nn.Module):
    """2-layer RGCN: a separate weight matrix per relation (mean-aggregated), plus a self weight."""

    def __init__(self, in_channels, num_relations, hidden_channels=64, dropout=0.3):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, 1, num_relations)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_type):
        x = F.relu(self.conv1(x, edge_index, edge_type))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index, edge_type).squeeze(-1)

In [ ]:
# ---- inlined from engines/graph/care_gnn.py (edit the module, then rebuild) ----
"""CARE-GNN (Dou et al., CIKM 2020, "Enhancing Graph Neural Network-based Fraud Detectors against
Camouflaged Fraudsters"), reimplemented from the paper's description for full-batch PyG.

What is implemented, per the paper:
  1. Label-aware similarity: a one-layer classifier on node features predicts P(illicit). The
     distance between two nodes is the gap between their predictions, and the classifier is trained
     with its own supervised loss added to the main loss (weight `lambda_simi`).
  2. Similarity-aware neighbor selection: per relation, each node keeps its ceil(p_r * degree_r)
     closest neighbors (at least one) and drops the rest before aggregating.
  3. Threshold update ("RL module"): after each training epoch, per relation, if the average distance
     of the selected neighbors went down, p_r += tau; otherwise p_r -= tau. A relation stops updating
     once the last 10 rewards sum to within +/-2. The paper frames this as a Bernoulli MDP; there is
     no learned policy network in the paper either, and none here.
  4. Relation-aware aggregation: mean over selected neighbors per relation, each relation's
     embedding scaled by its p_r, concatenated with the node's own embedding, then linear + ReLU.

Deliberate simplifications (stated so the result isn't mistaken for a line-by-line port):
  - Similarity is computed once per forward pass from the *input features*, and the same neighbor
    selection is used for both layers. The paper recomputes distances per layer from layer embeddings.
  - Two aggregation layers (to match the 2-hop depth of the other GNNs) + a linear classifier.
  - Imbalance is handled with the same class-weighted BCE as every other model here, instead of the
    paper's minority-class-balanced mini-batch sampling. Training is full-batch.
  - Relations come from `relations.build_relations` (payment direction), because Elliptic has no
    domain relation types like the paper's datasets (e.g. Yelp R-U-R / R-T-R / R-S-R).
"""
import math

import torch
import torch.nn.functional as F
from torch_geometric.utils import scatter


def top_p_per_target(dst, dist, p, num_nodes):
    """Boolean keep-mask over edges: for every target node, keep the ceil(p * in-degree) incoming
    edges with the smallest distance (always at least one when the node has any incoming edges)."""
    num_edges = dst.numel()
    if num_edges == 0:
        return torch.zeros(0, dtype=torch.bool, device=dst.device)
    order = torch.argsort(dist, stable=True)
    order = order[torch.argsort(dst[order], stable=True)]  # grouped by target, ascending distance
    dst_sorted = dst[order]
    counts = torch.bincount(dst_sorted, minlength=num_nodes)
    starts = torch.cumsum(counts, 0) - counts
    rank = torch.arange(num_edges, device=dst.device) - starts[dst_sorted]
    k = torch.clamp(torch.ceil(p * counts.float()), min=1).long()
    keep = torch.zeros(num_edges, dtype=torch.bool, device=dst.device)
    keep[order] = rank < k[dst_sorted]
    return keep


class CARELayer(torch.nn.Module):
    def __init__(self, in_channels, out_channels, num_relations):
        super().__init__()
        self.lin = torch.nn.Linear(in_channels * (num_relations + 1), out_channels)

    def forward(self, h, edge_index, edge_type, keep, thresholds):
        parts = [h]
        src, dst = edge_index
        for r in range(thresholds.numel()):
            sel = (edge_type == r) & keep
            agg = scatter(h[src[sel]], dst[sel], dim=0, dim_size=h.size(0), reduce="mean")
            parts.append(thresholds[r] * agg)
        return F.relu(self.lin(torch.cat(parts, dim=-1)))


class CAREGNN(torch.nn.Module):
    def __init__(self, in_channels, num_relations, hidden_channels=64, dropout=0.3,
                 lambda_simi=2.0, init_threshold=0.5, step_size=0.02, convergence_window=10):
        super().__init__()
        self.simi = torch.nn.Linear(in_channels, 1)
        self.layer1 = CARELayer(in_channels, hidden_channels, num_relations)
        self.layer2 = CARELayer(hidden_channels, hidden_channels, num_relations)
        self.classifier = torch.nn.Linear(hidden_channels, 1)
        self.dropout = dropout
        self.lambda_simi = lambda_simi
        self.step_size = step_size
        self.convergence_window = convergence_window
        self.num_relations = num_relations
        self.register_buffer("thresholds", torch.full((num_relations,), float(init_threshold)))

        self._simi_logits = None
        self._epoch_avg_dist = None
        self._prev_avg_dist = [None] * num_relations
        self.rewards = [[] for _ in range(num_relations)]
        self.converged = [False] * num_relations
        self.threshold_history = []
        self.last_keep = None

    @torch.no_grad()
    def _select(self, simi_logits, edge_index, edge_type):
        prob = torch.sigmoid(simi_logits).squeeze(-1)
        src, dst = edge_index
        # Distance between the predicted class distributions [p, 1-p]: L1 = 2|p_i - p_j|; the
        # constant factor doesn't change the ranking, so it is dropped.
        dist = (prob[src] - prob[dst]).abs()
        keep = torch.zeros_like(dst, dtype=torch.bool)
        avg_dist = []
        for r in range(self.num_relations):
            idx = (edge_type == r).nonzero(as_tuple=True)[0]
            k = top_p_per_target(dst[idx], dist[idx], self.thresholds[r], prob.numel())
            keep[idx[k]] = True
            avg_dist.append(float(dist[idx[k]].mean()) if k.any() else math.nan)
        return keep, avg_dist

    def forward(self, x, edge_index, edge_type):
        simi_logits = self.simi(x)
        keep, avg_dist = self._select(simi_logits, edge_index, edge_type)
        if self.training:
            self._simi_logits = simi_logits
            self._epoch_avg_dist = avg_dist
        self.last_keep = keep

        h = self.layer1(x, edge_index, edge_type, keep, self.thresholds)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.layer2(h, edge_index, edge_type, keep, self.thresholds)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.classifier(h).squeeze(-1)

    def auxiliary_loss(self, y, train_mask, pos_weight):
        """Supervised loss for the label-aware similarity classifier (called by the training loop)."""
        logits = self._simi_logits.squeeze(-1)[train_mask]
        return self.lambda_simi * F.binary_cross_entropy_with_logits(
            logits, y[train_mask].float(), pos_weight=pos_weight
        )

    @torch.no_grad()
    def on_epoch_end(self):
        """The paper's threshold update, driven by the average selected-neighbor distance."""
        for r, cur in enumerate(self._epoch_avg_dist):
            if self.converged[r] or math.isnan(cur):
                continue
            prev = self._prev_avg_dist[r]
            self._prev_avg_dist[r] = cur
            if prev is None:
                continue
            reward = 1 if prev - cur >= 0 else -1
            self.thresholds[r] = torch.clamp(self.thresholds[r] + reward * self.step_size, 0.0, 1.0)
            self.rewards[r].append(reward)
            window = self.rewards[r][-self.convergence_window:]
            if len(window) == self.convergence_window and abs(sum(window)) <= 2:
                self.converged[r] = True
        self.threshold_history.append([float(t) for t in self.thresholds])

    def diagnostics(self):
        return {
            "final_thresholds": [float(t) for t in self.thresholds],
            "converged": list(self.converged),
            "threshold_updates": [len(r) for r in self.rewards],
            "last_avg_selected_distance": self._epoch_avg_dist,
        }

In [ ]:
# ---- inlined from engines/graph/train.py (edit the module, then rebuild) ----
"""Full-batch training loop shared by every GNN (same protocol as notebook 01's train_model_full_batch)."""
import math

import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score


def train_full_batch(model, optimizer, forward, y, train_mask, pos_weight, epochs,
                     val_mask=None, patience=None, verbose=False):
    """Class-weighted BCE on `train_mask`. `forward(model)` returns one logit per node.

    With `val_mask`, keeps the state with the best validation PR-AUC and optionally early-stops after
    `patience` epochs without improvement. Models may define `auxiliary_loss(y, train_mask,
    pos_weight)` (added to the loss) and `on_epoch_end()` (called after each optimizer step)."""
    loss_history = []
    best_val, best_state, best_epoch, since_improve = -1.0, None, 0, 0
    epoch = 0
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = forward(model)
        loss = F.binary_cross_entropy_with_logits(out[train_mask], y[train_mask].float(), pos_weight=pos_weight)
        if hasattr(model, "auxiliary_loss"):
            loss = loss + model.auxiliary_loss(y, train_mask, pos_weight)
        loss.backward()
        optimizer.step()
        loss_value = loss.item()
        if not math.isfinite(loss_value):
            raise FloatingPointError(f"training diverged at epoch {epoch} (loss={loss_value})")
        loss_history.append(loss_value)
        if hasattr(model, "on_epoch_end"):
            model.on_epoch_end()

        if val_mask is None:
            continue
        val_pr_auc = pr_auc(predict_probs(model, forward), y, val_mask)
        if val_pr_auc > best_val:
            best_val, best_epoch, since_improve = val_pr_auc, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since_improve += 1
        if patience is not None and since_improve >= patience:
            if verbose:
                print(f"  early stopping at epoch {epoch} (best val PR-AUC {best_val:.4f} at epoch {best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return {
        "loss_history": loss_history,
        "best_val_pr_auc": best_val if val_mask is not None else None,
        "best_epoch": best_epoch if val_mask is not None else epoch,
        "epochs_run": epoch,
    }


@torch.no_grad()
def predict_probs(model, forward):
    model.eval()
    return torch.sigmoid(forward(model)).detach().cpu()


def pr_auc(probs, y, mask):
    mask = mask.cpu()
    return float(average_precision_score(y.cpu()[mask].numpy(), probs[mask].numpy()))

In [ ]:
# ---- inlined from engines/graph/evaluate.py (edit the module, then rebuild) ----
"""Camouflage subsets (same definitions as Phase 1C) and threshold-matched recall for cross-model comparison."""
import numpy as np
import torch
from sklearn.metrics import average_precision_score
from torch_geometric.utils import to_undirected


CAMOUFLAGE_DEFINITION_NAMES = ("mostly_licit_or_unknown", "zero_illicit_neighbors")
MIN_SUBSET_SIZE = 20


def neighbor_label_stats(edge_index, y, num_nodes):
    """Per node: number of real (non-self) undirected neighbors, and how many are labeled illicit."""
    src, dst = to_undirected(edge_index.cpu())
    not_self = src != dst
    is_illicit = (y.cpu() == ILLICIT).long()
    real_degree = torch.zeros(num_nodes, dtype=torch.long)
    real_degree.scatter_add_(0, dst[not_self], torch.ones(int(not_self.sum()), dtype=torch.long))
    illicit_neighbors = torch.zeros(num_nodes, dtype=torch.long)
    illicit_neighbors.scatter_add_(0, dst[not_self], is_illicit[src[not_self]])
    return real_degree, illicit_neighbors


def camouflage_subsets(edge_index, y, test_mask, num_nodes):
    """Illicit test nodes split into camouflaged / non-camouflaged under both Phase 1C definitions.
    Isolated illicit nodes belong to neither side (they aren't hiding among anything)."""
    real_degree, illicit_neighbors = neighbor_label_stats(edge_index, y, num_nodes)
    frac = illicit_neighbors.float() / real_degree.clamp(min=1).float()
    has_neighbors = real_degree > 0
    definitions = {
        "mostly_licit_or_unknown": has_neighbors & (frac < 0.5),
        "zero_illicit_neighbors": has_neighbors & (illicit_neighbors == 0),
    }
    y_cpu, test_cpu = y.cpu(), test_mask.cpu()
    illicit_test = (test_cpu & (y_cpu == ILLICIT)).nonzero(as_tuple=True)[0]
    licit_test = (test_cpu & (y_cpu == LICIT)).nonzero(as_tuple=True)[0]
    subsets = {"all_illicit": illicit_test}
    for name, mask in definitions.items():
        subsets[f"{name}/camouflaged"] = illicit_test[mask[illicit_test]]
        subsets[f"{name}/non_camouflaged"] = illicit_test[(~mask[illicit_test]) & has_neighbors[illicit_test]]
    return subsets, licit_test


def threshold_at_fpr(probs, licit_idx, target_fpr):
    """Score threshold at which roughly `target_fpr` of licit test nodes are flagged. Comparing
    models' recall at the same false-positive rate removes calibration differences that make a
    fixed 0.5 cutoff unfair across models."""
    negatives = probs[licit_idx].double().numpy()
    threshold = float(np.quantile(negatives, 1.0 - target_fpr))
    actual_fpr = float((negatives > threshold).mean())
    return threshold, actual_fpr


def subset_metrics(probs, idx, licit_idx, threshold_matched):
    if len(idx) == 0:
        return {"n": 0, "recall_at_0.5": None, "recall_at_matched_fpr": None,
                "pr_auc_vs_licit": None, "pr_auc_chance_baseline": None}
    pos = probs[idx]
    scores = torch.cat([pos, probs[licit_idx]]).numpy()
    labels = np.concatenate([np.ones(len(idx)), np.zeros(len(licit_idx))])
    return {
        "n": int(len(idx)),
        "recall_at_0.5": float((pos > 0.5).float().mean()),
        "recall_at_matched_fpr": float((pos > threshold_matched).float().mean()),
        "pr_auc_vs_licit": float(average_precision_score(labels, scores)),
        "pr_auc_chance_baseline": float(len(idx) / (len(idx) + len(licit_idx))),
    }


def camouflage_report(probs, subsets, licit_idx, target_fpr=0.05):
    threshold, actual_fpr = threshold_at_fpr(probs, licit_idx, target_fpr)
    report = {
        "target_fpr": target_fpr,
        "matched_threshold": threshold,
        "actual_fpr_on_licit_test": actual_fpr,
        "subsets": {name: subset_metrics(probs, idx, licit_idx, threshold) for name, idx in subsets.items()},
    }
    return report

## 1. Setup and reproduction check

In [ ]:
data = make_synthetic_elliptic() if SMOKE_TEST else load_elliptic(DATA_ROOT)
n_nodes, n_features = data.num_nodes, data.num_node_features
train_mask, val_mask, test_mask = temporal_masks(data)
counts = list(split_counts(train_mask, val_mask, test_mask))
pos_weight = class_pos_weight(data.y, train_mask)
time_stats = edge_time_stats(data.edge_index, data.time_step)

x_d, ei_d, y_d = data.x.to(DEVICE), data.edge_index.to(DEVICE), data.y.to(DEVICE)
train_d, val_d, test_d = train_mask.to(DEVICE), val_mask.to(DEVICE), test_mask.to(DEVICE)
pos_weight_t = torch.tensor([pos_weight], device=DEVICE, dtype=torch.float32)


def forward_plain(model):
    return model(x_d, ei_d)


def check_against_reference(name, value, tolerance):
    ref = REFERENCE[name]
    if SMOKE_TEST:
        return {"this_run": value, "reference": ref, "match": None, "note": "skipped (synthetic smoke data)"}
    if isinstance(ref, list):
        match = list(value) == ref
    else:
        match = abs(value - ref) <= tolerance
    status = "OK" if match else "MISMATCH"
    print(f"  reproduction {name}: this run {value} vs reference {ref} -> {status}")
    return {"this_run": value, "reference": ref, "tolerance": tolerance, "match": bool(match)}


print(f"nodes {n_nodes} | features {n_features} | split (train/val/test) {counts} | pos_weight {pos_weight:.4f}")
print(f"edges: {time_stats['edges']} total, {time_stats['same_step']} within a time step, "
      f"{time_stats['cross_step']} crossing time steps")
REPRODUCTION = {
    "split_counts": check_against_reference("split_counts", counts, 0),
    "pos_weight": check_against_reference("pos_weight", pos_weight, 1e-9),
}
save_progress("setup", {"nodes": n_nodes, "features": n_features, "split_counts": counts,
                        "pos_weight": pos_weight, "edge_time_stats": time_stats})

MODEL_RESULTS, PROBS, MODELS = {}, {}, {}


def record_model(name, family, protocol, probs, model=None, extra=None):
    test_score = pr_auc(probs, data.y, test_mask)
    val_score = pr_auc(probs, data.y, val_mask)
    MODEL_RESULTS[name] = {"family": family, "protocol": protocol, "pr_auc_test": test_score,
                           "pr_auc_val": val_score, **(extra or {})}
    PROBS[name] = probs
    if model is not None:
        MODELS[name] = model
    print(f"{name:28s} test PR-AUC {test_score:.4f} | val {val_score:.4f} | {protocol}")
    save_progress("models", MODEL_RESULTS)
    return test_score

In [ ]:
# XGBoost baselines, refit from the stored Phase 1 / 1B best params.
X_tier1 = data.x.numpy()
t0 = time.time()
tier2 = tier2_graph_features(data.edge_index, n_nodes)
print(f"Tier-2 graph features computed in {time.time() - t0:.0f}s")
X_tier12 = np.concatenate([X_tier1, tier2], axis=1)
train_np, y_np = train_mask.numpy(), data.y.numpy()

for name, X, params, protocol in [
    ("xgboost_tier1", X_tier1, XGB_TIER1_PARAMS, "raw features; params from 25-trial Optuna search (Phase 1)"),
    ("xgboost_tier12", X_tier12, XGB_TIER12_PARAMS, "raw + Tier-2 graph features; params from 25-trial Optuna search (Phase 1B)"),
]:
    model = fit_xgb(X[train_np], y_np[train_np], params, pos_weight, SEED)
    probs = torch.from_numpy(model.predict_proba(X)[:, 1]).float()
    score = record_model(name, "xgboost", protocol, probs, model=model)
    REPRODUCTION[name] = check_against_reference(name, score, 0.002)

save_progress("reproduction", REPRODUCTION)

In [ ]:
# Best GNN before Phase 1D: GraphSAGE with the Phase 1C wide-search config, refit with notebook 01's
# protocol (early stopping on val, patience 15, 150 epochs max). This is the "before" model for CARE-GNN.
torch.manual_seed(SEED)
sage = GraphSAGEFlex(n_features, GRAPHSAGE_WIDE_PARAMS["hidden_dim"], GRAPHSAGE_WIDE_PARAMS["dropout"]).to(DEVICE)
opt = torch.optim.Adam(sage.parameters(), lr=GRAPHSAGE_WIDE_PARAMS["lr"], weight_decay=5e-4)
fit = train_full_batch(sage, opt, forward_plain, y_d, train_d, pos_weight_t, epochs=REPRO_REFIT_EPOCHS,
                       val_mask=val_d, patience=15, verbose=True)
score = record_model("graphsage_wide_search", "graphsage",
                     "directed edges; Phase 1C 50-trial search config; early-stopped refit",
                     predict_probs(sage, forward_plain), model=sage,
                     extra={"params": GRAPHSAGE_WIDE_PARAMS, "best_epoch": fit["best_epoch"]})
# GPU nondeterminism moved earlier GNN re-runs by ~0.001; allow more slack for a 256-dim model.
REPRODUCTION["graphsage_wide_search"] = check_against_reference("graphsage_wide_search", score, 0.02)
save_progress("reproduction", REPRODUCTION)

## 2. RGCN with label-free relation types

Elliptic's edges are untyped, so relation types have to be constructed. The spec's candidates, and
why they were or weren't used (details in `docs/design_decisions.md`):
- **Same-step vs. cross-step edges**: only meaningful if cross-step edges exist; the edge count
  above shows how many do.
- **Labeled-adjacent vs. unknown-only**: rejected. Typing an edge by its endpoints' labels leaks
  test labels into the graph, and because test-period labels are hidden at training time, the
  relation mix would differ between the train and test periods.
- **Payment direction** (used): upstream vs. downstream neighbor. Label-free, meaningful for money
  flow, and adds the downstream messages the notebook 01 GNNs never saw.
- **Direction x hub source** (used as a variant): splits out messages from the top-1% degree nodes.

All RGCNs use the "defaults" protocol (hidden 64, dropout 0.3, lr 0.01, 40 epochs), the same as
GraphSAGE/GAT/GIN defaults in notebook 01. The `single` variant is the control: same edges in both
directions, no typing -- so `direction` minus `single` isolates the effect of relation typing.

In [ ]:
RELATIONS = {}
for scheme in RELATION_SCHEMES:
    ei_r, et_r, names, info = build_relations(data.edge_index, n_nodes, scheme)
    RELATIONS[scheme] = {"edge_index": ei_r.to(DEVICE), "edge_type": et_r.to(DEVICE), "names": names,
                         "info": info, "counts": relation_counts(et_r, names)}
    print(f"{scheme:14s} relations: {RELATIONS[scheme]['counts']} {info}")


def forward_relational(scheme):
    rel = RELATIONS[scheme]
    return lambda model: model(x_d, rel["edge_index"], rel["edge_type"])


for scheme in RELATION_SCHEMES:
    torch.manual_seed(SEED)
    rgcn = RGCN(n_features, len(RELATIONS[scheme]["names"]), hidden_channels=64, dropout=0.3).to(DEVICE)
    opt = torch.optim.Adam(rgcn.parameters(), lr=0.01, weight_decay=5e-4)
    fit = train_full_batch(rgcn, opt, forward_relational(scheme), y_d, train_d, pos_weight_t, epochs=DEFAULT_EPOCHS)
    record_model(f"rgcn_{scheme}", "rgcn", "defaults (hidden 64, dropout 0.3, lr 0.01, 40 epochs)",
                 predict_probs(rgcn, forward_relational(scheme)), model=rgcn,
                 extra={"relation_scheme": scheme, "relation_counts": RELATIONS[scheme]["counts"],
                        "relation_info": RELATIONS[scheme]["info"],
                        "train_loss_first_last": [fit["loss_history"][0], fit["loss_history"][-1]]})

typing_effect = MODEL_RESULTS["rgcn_direction"]["pr_auc_test"] - MODEL_RESULTS["rgcn_single"]["pr_auc_test"]
hub_effect = MODEL_RESULTS["rgcn_direction_hub"]["pr_auc_test"] - MODEL_RESULTS["rgcn_direction"]["pr_auc_test"]
best_rgcn = max((n for n in MODEL_RESULTS if n.startswith("rgcn_")), key=lambda n: MODEL_RESULTS[n]["pr_auc_test"])
rgcn_summary = {
    "relation_typing_effect_direction_minus_single": typing_effect,
    "hub_split_effect_direction_hub_minus_direction": hub_effect,
    "best_rgcn": best_rgcn,
    "best_rgcn_pr_auc_test": MODEL_RESULTS[best_rgcn]["pr_auc_test"],
    "reference_graphsage_default_directed_same_protocol": REFERENCE["graphsage_default_directed"],
    "reference_graphsage_wide_search": REFERENCE["graphsage_wide_search"],
    "cross_step_edges": time_stats["cross_step"],
}
print(f"\nrelation typing (direction - single): {typing_effect:+.4f} | hub split (direction_hub - direction): "
      f"{hub_effect:+.4f} | best RGCN: {best_rgcn} ({MODEL_RESULTS[best_rgcn]['pr_auc_test']:.4f})")
save_progress("rgcn", rgcn_summary)

## 3. CARE-GNN

Implementation and every simplification are documented in the `care_gnn.py` cell above and in
`docs/design_decisions.md`. In short: the paper's label-aware similarity, top-p neighbor
selection per relation, the paper's reward-based threshold update (+/- tau per epoch, stop after
10 epochs of rewards summing within +/-2), and relation-aware aggregation. Simplified: similarity
from input features shared by both layers, class-weighted loss instead of balanced sampling,
full-batch, and payment-direction relations.

Relation scheme fixed to `direction` in advance (not picked from the RGCN test results above).
Trained with early stopping on val, like the wide-search GraphSAGE it is compared against.

In [ ]:
CARE_SCHEME = "direction"
CARE_PARAMS = {"hidden_dim": 64, "dropout": 0.3, "lr": 0.01, "lambda_simi": 2.0,
               "init_threshold": 0.5, "step_size": 0.02}

torch.manual_seed(SEED)
care = CAREGNN(n_features, len(RELATIONS[CARE_SCHEME]["names"]), hidden_channels=CARE_PARAMS["hidden_dim"],
               dropout=CARE_PARAMS["dropout"], lambda_simi=CARE_PARAMS["lambda_simi"],
               init_threshold=CARE_PARAMS["init_threshold"], step_size=CARE_PARAMS["step_size"]).to(DEVICE)
opt = torch.optim.Adam(care.parameters(), lr=CARE_PARAMS["lr"], weight_decay=5e-4)
t0 = time.time()
fit = train_full_batch(care, opt, forward_relational(CARE_SCHEME), y_d, train_d, pos_weight_t,
                       epochs=REFIT_MAX_EPOCHS, val_mask=val_d, patience=15, verbose=True)
care_diag = {**care.diagnostics(), "best_epoch": fit["best_epoch"], "epochs_run": fit["epochs_run"],
             "thresholds_at_restored_best_epoch": [float(t) for t in care.thresholds],
             "threshold_history_every_10_epochs": care.threshold_history[::10],
             "train_seconds": round(time.time() - t0, 1)}
record_model("care_gnn", "care_gnn",
             "simplified CARE-GNN (see design_decisions.md); direction relations; early-stopped (patience 15)",
             predict_probs(care, forward_relational(CARE_SCHEME)), model=care,
             extra={"params": CARE_PARAMS, "diagnostics": care_diag})
print("CARE-GNN diagnostics:", {k: v for k, v in care_diag.items() if k != "threshold_history_every_10_epochs"})

In [ ]:
# Camouflage before/after at a matched false-positive rate. Same subset definitions as Phase 1C.
subsets, licit_test_idx = camouflage_subsets(data.edge_index, data.y, test_mask, n_nodes)
CAMOUFLAGE_MODELS = ["xgboost_tier12", "graphsage_wide_search", best_rgcn, "care_gnn"]
CAMO = {name: camouflage_report(PROBS[name], subsets, licit_test_idx, CAMOUFLAGE_TARGET_FPR)
        for name in CAMOUFLAGE_MODELS}

cols = ["all_illicit", "zero_illicit_neighbors/camouflaged", "zero_illicit_neighbors/non_camouflaged",
        "mostly_licit_or_unknown/camouflaged", "mostly_licit_or_unknown/non_camouflaged"]
print(f"recall at matched {CAMOUFLAGE_TARGET_FPR:.0%} FPR on licit test nodes (subset sizes: "
      + ", ".join(f"{c}={CAMO['care_gnn']['subsets'][c]['n']}" for c in cols) + ")")
print(f"{'model':24s}" + "".join(f"{c.split('/')[-1][:14]:>16s}" for c in cols) + "   actual FPR")
for name in CAMOUFLAGE_MODELS:
    row = CAMO[name]["subsets"]
    cells = "".join(f"{row[c]['recall_at_matched_fpr']:16.4f}" if row[c]["n"] else f"{'n/a':>16s}" for c in cols)
    print(f"{name:24s}{cells}   {CAMO[name]['actual_fpr_on_licit_test']:.4f}")


def recall_delta(a, b, subset):
    """Recall difference (a - b) on a subset at matched FPR, with a normal-approximation standard error."""
    ma, mb = CAMO[a]["subsets"][subset], CAMO[b]["subsets"][subset]
    if min(ma["n"], mb["n"]) < MIN_SUBSET_SIZE:
        return None
    ra, rb, n = ma["recall_at_matched_fpr"], mb["recall_at_matched_fpr"], ma["n"]
    se = math.sqrt(max(ra * (1 - ra), 1e-9) / n + max(rb * (1 - rb), 1e-9) / n)
    delta = ra - rb
    return {"delta": delta, "approx_se": se, "beyond_2se": abs(delta) > 2 * se}


# Does CARE-GNN's selector act on camouflaged fraud as intended? For illicit and licit test targets,
# the share of incoming edges kept, split by the neighbor's label.
predict_probs(care, forward_relational(CARE_SCHEME))  # refresh last_keep in eval mode
keep = care.last_keep.cpu()
src_r, dst_r = RELATIONS[CARE_SCHEME]["edge_index"].cpu()
selector_keep_rates = {}
for target_label, target_name in [(ILLICIT, "illicit_test_targets"), (LICIT, "licit_test_targets")]:
    target_sel = test_mask[dst_r] & (data.y[dst_r] == target_label)
    rates = {}
    for nb_label, nb_name in [(LICIT, "licit"), (ILLICIT, "illicit"), (UNKNOWN, "unknown")]:
        sel = target_sel & (data.y[src_r] == nb_label)
        rates[f"{nb_name}_neighbors"] = {"edges": int(sel.sum()),
                                         "kept_share": float(keep[sel].float().mean()) if sel.any() else None}
    selector_keep_rates[target_name] = rates
print("\nCARE-GNN selector keep rates:", json.dumps(selector_keep_rates, indent=1))

CAMO_COMPARISONS = {
    subset: {
        "care_minus_graphsage": recall_delta("care_gnn", "graphsage_wide_search", subset),
        "care_minus_xgboost_tier12": recall_delta("care_gnn", "xgboost_tier12", subset),
        "graphsage_minus_xgboost_tier12": recall_delta("graphsage_wide_search", "xgboost_tier12", subset),
    }
    for subset in ["all_illicit", "zero_illicit_neighbors/camouflaged", "mostly_licit_or_unknown/camouflaged"]
}


def describe_delta(d, a_name, b_name):
    if d is None:
        return f"{a_name} vs {b_name}: subset too small to compare"
    verdict = ("higher" if d["delta"] > 0 else "lower") if d["beyond_2se"] else "no clear difference (within ~2 SE)"
    return f"{a_name} vs {b_name}: {d['delta']:+.4f} (SE ~{d['approx_se']:.4f}) -> {verdict}"


camo_key = "zero_illicit_neighbors/camouflaged"
camouflage_finding = (
    f"On the camouflaged subset (illicit test nodes with no illicit-labeled neighbors, n="
    f"{CAMO['care_gnn']['subsets'][camo_key]['n']}), recall at a matched {CAMOUFLAGE_TARGET_FPR:.0%} FPR: "
    f"CARE-GNN {CAMO['care_gnn']['subsets'][camo_key]['recall_at_matched_fpr']:.4f}, "
    f"GraphSAGE (wide search) {CAMO['graphsage_wide_search']['subsets'][camo_key]['recall_at_matched_fpr']:.4f}, "
    f"XGBoost Tier-1+2 {CAMO['xgboost_tier12']['subsets'][camo_key]['recall_at_matched_fpr']:.4f}. "
    + describe_delta(CAMO_COMPARISONS[camo_key]["care_minus_graphsage"], "CARE-GNN", "GraphSAGE") + "; "
    + describe_delta(CAMO_COMPARISONS[camo_key]["care_minus_xgboost_tier12"], "CARE-GNN", "XGBoost Tier-1+2") + "."
)
print("\n" + camouflage_finding)
save_progress("camouflage_before_after", {
    "models": CAMO, "comparisons": CAMO_COMPARISONS, "selector_keep_rates": selector_keep_rates,
    "finding": camouflage_finding,
    "note": "Phase 1C reported recall at p>0.5 within one model; across models, recall is compared at a "
            "matched false-positive rate on licit test nodes because the models are calibrated differently.",
})

## 4. Leaderboard (models trained in this notebook)

In [ ]:
def print_leaderboard(results):
    rows = sorted(results.items(), key=lambda kv: kv[1]["pr_auc_test"], reverse=True)
    print(f"{'model':28s} {'family':10s} {'test PR-AUC':>11s} {'val PR-AUC':>10s}  protocol")
    for name, r in rows:
        print(f"{name:28s} {r['family']:10s} {r['pr_auc_test']:11.4f} {r['pr_auc_val']:10.4f}  {r['protocol']}")


print_leaderboard(MODEL_RESULTS)

## 5. Expanded search on the best GNN

"Best" = highest temporal-test PR-AUC among the GNNs above, as the spec defines it. Picking a model
by its test score makes that model's leaderboard number slightly optimistic; the 3-seed check below
and the separate validation scores are the guard against over-reading it. The search itself selects
on validation (time steps 31-35) only, with early stopping, as in every earlier phase.

In [ ]:
GNN_FAMILIES = {"graphsage", "rgcn", "care_gnn"}
gnn_names = [n for n, r in MODEL_RESULTS.items() if r["family"] in GNN_FAMILIES]
search_base = max(gnn_names, key=lambda n: MODEL_RESULTS[n]["pr_auc_test"])
search_family = MODEL_RESULTS[search_base]["family"]
search_scheme = MODEL_RESULTS[search_base].get("relation_scheme", CARE_SCHEME if search_family == "care_gnn" else None)
print(f"best GNN so far: {search_base} ({MODEL_RESULTS[search_base]['pr_auc_test']:.4f}) -> searching family "
      f"'{search_family}'" + (f" with relation scheme '{search_scheme}'" if search_scheme else ""))

SEARCH_SPACES = {
    "graphsage": {"hidden_dim": [128, 256, 512]},
    "rgcn": {"hidden_dim": [32, 64, 128, 256]},
    "care_gnn": {"hidden_dim": [32, 64, 128], "lambda_simi": (0.5, 4.0), "init_threshold": (0.2, 0.9),
                 "step_size": (0.005, 0.05)},
}
COMMON_RANGES = {"lr": (1e-4, 3e-2), "dropout": (0.1, 0.6), "max_epochs": (min(30, SEARCH_MAX_EPOCHS), SEARCH_MAX_EPOCHS)}


def suggest_params(trial, family):
    params = {
        "lr": trial.suggest_float("lr", *COMMON_RANGES["lr"], log=True),
        "dropout": trial.suggest_float("dropout", *COMMON_RANGES["dropout"]),
        "max_epochs": trial.suggest_int("max_epochs", *COMMON_RANGES["max_epochs"]),
        "hidden_dim": trial.suggest_categorical("hidden_dim", SEARCH_SPACES[family]["hidden_dim"]),
    }
    if family == "care_gnn":
        params["lambda_simi"] = trial.suggest_float("lambda_simi", *SEARCH_SPACES[family]["lambda_simi"])
        params["init_threshold"] = trial.suggest_float("init_threshold", *SEARCH_SPACES[family]["init_threshold"])
        params["step_size"] = trial.suggest_float("step_size", *SEARCH_SPACES[family]["step_size"], log=True)
    return params


def build_model(family, params, seed, scheme):
    torch.manual_seed(seed)
    if family == "graphsage":
        model, forward = GraphSAGEFlex(n_features, params["hidden_dim"], params["dropout"]), forward_plain
    elif family == "rgcn":
        model = RGCN(n_features, len(RELATIONS[scheme]["names"]), params["hidden_dim"], params["dropout"])
        forward = forward_relational(scheme)
    else:
        model = CAREGNN(n_features, len(RELATIONS[scheme]["names"]), params["hidden_dim"], params["dropout"],
                        lambda_simi=params["lambda_simi"], init_threshold=params["init_threshold"],
                        step_size=params["step_size"])
        forward = forward_relational(scheme)
    model = model.to(DEVICE)
    return model, forward, torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=5e-4)


def objective(trial):
    params = suggest_params(trial, search_family)
    model, forward, optimizer = build_model(search_family, params, SEED, search_scheme)
    try:
        fit = train_full_batch(model, optimizer, forward, y_d, train_d, pos_weight_t, epochs=params["max_epochs"],
                               val_mask=val_d, patience=10)
    except (FloatingPointError, torch.cuda.OutOfMemoryError) as e:
        print(f"  trial {trial.number} failed: {type(e).__name__}: {e}")
        raise optuna.TrialPruned()
    finally:
        del model, optimizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return fit["best_val_pr_auc"]


def progress_callback(study, trial):
    done = len(study.trials)
    if done % 10 == 0 or done == SEARCH_TRIALS:
        n_ok = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
        best = f"{study.best_value:.4f}" if n_ok else "n/a"
        print(f"  {done} trials ({n_ok} completed) | best val PR-AUC {best} | elapsed {time.time() - search_start:.0f}s")


search_start = time.time()
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=SEARCH_TRIALS, timeout=SEARCH_TIMEOUT_SEC, callbacks=[progress_callback])
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if not completed:
    save_progress("expanded_search", {"base_model": search_base, "family": search_family,
                                      "trials_total": len(study.trials), "trials_completed": 0})
    raise RuntimeError("every search trial failed (see messages above) -- log this as a real bug; tasks 1-4 are saved")
search_best = study.best_params

at_boundary = []
for key, value in search_best.items():
    if key in SEARCH_SPACES[search_family] and isinstance(SEARCH_SPACES[search_family][key], list):
        if value == max(SEARCH_SPACES[search_family][key]):
            at_boundary.append(f"{key}={value} (max option)")
        continue
    lo, hi = COMMON_RANGES.get(key) or SEARCH_SPACES[search_family][key]
    if key in ("lr", "step_size"):
        near_lo, near_hi = value <= lo * 1.5, value >= hi / 1.5
    else:
        span = hi - lo
        near_lo, near_hi = value <= lo + 0.05 * span, value >= hi - 0.05 * span
    if near_lo or near_hi:
        at_boundary.append(f"{key}={value:.4g} (near {'lower' if near_lo else 'upper'} bound)")

print(f"search: {len(completed)}/{len(study.trials)} trials completed in {time.time() - search_start:.0f}s "
      f"(cap {SEARCH_TRIALS} trials / {SEARCH_TIMEOUT_SEC}s)")
print(f"best params: {search_best} | best val PR-AUC {study.best_value:.4f}")
print("at a search-space boundary:", at_boundary or "none")

final_model, final_forward, final_opt = build_model(search_family, search_best, SEED, search_scheme)
fit = train_full_batch(final_model, final_opt, final_forward, y_d, train_d, pos_weight_t, epochs=REFIT_MAX_EPOCHS,
                       val_mask=val_d, patience=15, verbose=True)
final_name = f"{search_family}_expanded_search"
record_model(final_name, search_family,
             f"{len(completed)}-trial Optuna search (val-selected, early stopping); early-stopped refit",
             predict_probs(final_model, final_forward), model=final_model,
             extra={"params": search_best, "relation_scheme": search_scheme, "best_epoch": fit["best_epoch"]})
search_change = MODEL_RESULTS[final_name]["pr_auc_test"] - MODEL_RESULTS[search_base]["pr_auc_test"]
save_progress("expanded_search", {
    "base_model": search_base, "family": search_family, "relation_scheme": search_scheme,
    "trials_requested": SEARCH_TRIALS, "trials_completed": len(completed), "trials_total": len(study.trials),
    "timeout_sec": SEARCH_TIMEOUT_SEC, "seconds": round(time.time() - search_start, 1),
    "best_params": search_best, "best_val_pr_auc": study.best_value, "at_boundary": at_boundary,
    "pr_auc_test": MODEL_RESULTS[final_name]["pr_auc_test"], "change_vs_base": search_change,
    "search_space": {**{k: list(v) for k, v in COMMON_RANGES.items()}, **SEARCH_SPACES[search_family]},
})

## 6. Robustness check: 3 seeds on the final best GNN

In [ ]:
final_best_gnn = max((n for n, r in MODEL_RESULTS.items() if r["family"] in GNN_FAMILIES),
                     key=lambda n: MODEL_RESULTS[n]["pr_auc_test"])
print(f"final best GNN by test PR-AUC: {final_best_gnn} ({MODEL_RESULTS[final_best_gnn]['pr_auc_test']:.4f})")

# Re-run the winner with exactly the protocol that produced its leaderboard number.
if final_best_gnn == final_name:
    robust_family, robust_params, robust_scheme = search_family, search_best, search_scheme
    robust_epochs, robust_early_stop = REFIT_MAX_EPOCHS, True
elif final_best_gnn == "care_gnn":
    robust_family, robust_params, robust_scheme = "care_gnn", CARE_PARAMS, CARE_SCHEME
    robust_epochs, robust_early_stop = REFIT_MAX_EPOCHS, True
elif final_best_gnn == "graphsage_wide_search":
    robust_family, robust_params, robust_scheme = "graphsage", GRAPHSAGE_WIDE_PARAMS, None
    robust_epochs, robust_early_stop = REPRO_REFIT_EPOCHS, True
else:  # an untuned RGCN variant won: defaults protocol, fixed epochs, no early stopping
    robust_family, robust_scheme = "rgcn", MODEL_RESULTS[final_best_gnn]["relation_scheme"]
    robust_params = {"hidden_dim": 64, "dropout": 0.3, "lr": 0.01}
    robust_epochs, robust_early_stop = DEFAULT_EPOCHS, False

robust_scores = []
for seed in ROBUSTNESS_SEEDS:
    model, forward, optimizer = build_model(robust_family, robust_params, seed, robust_scheme)
    train_full_batch(model, optimizer, forward, y_d, train_d, pos_weight_t, epochs=robust_epochs,
                     val_mask=val_d if robust_early_stop else None, patience=15 if robust_early_stop else None)
    robust_scores.append(pr_auc(predict_probs(model, forward), data.y, test_mask))
    print(f"  seed {seed}: test PR-AUC {robust_scores[-1]:.4f}")
    del model, optimizer
robust_mean, robust_std = float(np.mean(robust_scores)), float(np.std(robust_scores))
print(f"{final_best_gnn}: {robust_mean:.4f} +/- {robust_std:.4f} across seeds {ROBUSTNESS_SEEDS}")
save_progress("robustness_best_gnn", {"model": final_best_gnn, "family": robust_family, "params": robust_params,
                                      "relation_scheme": robust_scheme, "max_epochs": robust_epochs,
                                      "early_stopping": robust_early_stop, "seeds": ROBUSTNESS_SEEDS,
                                      "scores": robust_scores, "mean": robust_mean, "std": robust_std})

In [ ]:
# Camouflage check on the final best GNN too, if the expanded search produced a new one.
if final_best_gnn not in CAMO:
    CAMO[final_best_gnn] = camouflage_report(PROBS[final_best_gnn], subsets, licit_test_idx, CAMOUFLAGE_TARGET_FPR)
    final_camo = CAMO[final_best_gnn]["subsets"][camo_key]
    print(f"{final_best_gnn} camouflaged-subset recall at matched FPR: {final_camo['recall_at_matched_fpr']:.4f}")
    save_progress("camouflage_final_best_gnn", CAMO[final_best_gnn])

## 7. Closing analysis and save

In [ ]:
# Compare against the XGBoost Tier-1+2 refit from THIS run (guaranteed same setup); the 3-seed mean from
# notebook 01 is quoted alongside for its variance.
xgb_this_run = MODEL_RESULTS["xgboost_tier12"]["pr_auc_test"]
gap_to_xgb = robust_mean - xgb_this_run
standing_finding_changed = robust_mean >= xgb_this_run
care_vs_sage = CAMO_COMPARISONS[camo_key]["care_minus_graphsage"]
care_vs_xgb = CAMO_COMPARISONS[camo_key]["care_minus_xgboost_tier12"]

overall = (
    f"Overall: the best GNN after Phase 1D is {final_best_gnn}, at {robust_mean:.4f} +/- {robust_std:.4f} "
    f"test PR-AUC over 3 seeds, vs XGBoost on raw + classical graph features at {xgb_this_run:.4f} in this run "
    f"({REFERENCE['xgboost_tier12_3seed_mean']:.4f} +/- {REFERENCE['xgboost_tier12_3seed_std']:.4f} over 3 seeds "
    f"in notebook 01), a gap of {gap_to_xgb:+.4f}. "
    + ("The standing finding CHANGES: a message-passing GNN now matches or beats tuned XGBoost."
       if standing_finding_changed else
       "The standing finding holds: message-passing GNNs still trail tuned XGBoost on this dataset.")
)
if care_vs_sage is None or care_vs_xgb is None:
    targeted = "The camouflaged subset was too small for a before/after comparison."
else:
    care_helped = care_vs_sage["delta"] > 0 and care_vs_sage["beyond_2se"]
    care_hurt = care_vs_sage["delta"] < 0 and care_vs_sage["beyond_2se"]
    care_matches_xgb = not (care_vs_xgb["delta"] < 0 and care_vs_xgb["beyond_2se"])
    targeted = (
        "Targeted defense: "
        + ("CARE-GNN's neighbor selection improved recall on the camouflaged subset over the pre-1D best GNN "
           if care_helped else
           "CARE-GNN's neighbor selection made camouflaged-subset recall worse than the pre-1D best GNN "
           if care_hurt else
           "CARE-GNN's neighbor selection did not measurably change camouflaged-subset recall vs the pre-1D best GNN ")
        + f"({care_vs_sage['delta']:+.4f}, SE ~{care_vs_sage['approx_se']:.4f}), and on that subset it "
        + ("matches or beats XGBoost" if care_matches_xgb else "still trails XGBoost")
        + f" ({care_vs_xgb['delta']:+.4f}), at a matched {CAMOUFLAGE_TARGET_FPR:.0%} false-positive rate."
    )
typing = (
    f"Relation typing: RGCN with payment-direction relations vs the same edges untyped: "
    f"{rgcn_summary['relation_typing_effect_direction_minus_single']:+.4f} test PR-AUC; "
    f"best RGCN {best_rgcn} at {rgcn_summary['best_rgcn_pr_auc_test']:.4f}."
)
closing_paragraph = " ".join([overall, targeted, typing])
print(closing_paragraph)

reproduction_ok = all(v.get("match") is not False for v in REPRODUCTION.values())
if not reproduction_ok and not SMOKE_TEST:
    closing_paragraph += (" WARNING: at least one reproduction check against notebook 01 failed (see "
                          "'reproduction'), so this notebook's numbers may not be directly comparable to earlier phases.")
    print("\nWARNING: reproduction mismatch -- see RESULTS['reproduction']")

save_progress("closing_paragraph", closing_paragraph)
save_progress("standing_finding_changed", bool(standing_finding_changed))

for name, model in MODELS.items():
    if isinstance(model, torch.nn.Module):
        torch.save(model.state_dict(), OUTPUT_DIR / f"phase1d_{name}.pt")
    else:
        model.save_model(str(OUTPUT_DIR / f"phase1d_{name}.json"))
print(f"saved {len(MODELS)} models to {OUTPUT_DIR}")
print_leaderboard(MODEL_RESULTS)
print(f"\nFull results: {RESULTS_PATH}")
print(json.dumps(RESULTS, indent=2))